# Example 11.3.1.1 / 1: Pass a List of Values to a Statistics Function

Companion notebook for **11.3.1.1 GSL Function Reference** in the AMPL lab Documentation, section "Pass a List of Values to a Statistics Function". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Pass a List of Values to a Statistics Function

A statistics function takes its dataset as an ordinary argument list, which you usually write as an
indexed expression such as `{p in PROD} rate[p]`. Functions that compare two datasets, such as
`gsl_stats_correlation`, split the argument list into two halves. Extra scalar arguments, such as a
known `mean` or the quantile fraction `f`, come after the data.

The steel production data from chapter 1, summarized with three statistics. The functions are
evaluated when the defined parameters are first used, so they work in the model or in a script:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
load amplgsl.dll;
function gsl_stats_mean;         # arithmetic mean of its arguments
function gsl_stats_sd;           # sample standard deviation
function gsl_stats_correlation;  # first half of the arguments versus second half

set PROD;                  # products
param rate {PROD} > 0;     # tons produced per hour
param profit {PROD};       # profit per ton

param mean_rate = gsl_stats_mean({p in PROD} rate[p]);  # average production rate
param sd_rate = gsl_stats_sd({p in PROD} rate[p]);      # spread of the rates
param rate_profit_corr =                                # do faster products earn less?
    gsl_stats_correlation({p in PROD} rate[p], {p in PROD} profit[p]);

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}

In [5]:
# run.py
print("mean rate:", ampl.param["mean_rate"].value())
print("sd of rate:", round(ampl.param["sd_rate"].value(), 4))
print("correlation:", round(ampl.param["rate_profit_corr"].value(), 4))

mean rate: 166.66666666666666
sd of rate: 30.5505
correlation: -0.9897


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param:   rate  profit :=
  bands   200    25
  coils   140    30
  plate   160    29 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display mean_rate, sd_rate, rate_profit_corr;

mean_rate = 166.667
sd_rate = 30.5505
rate_profit_corr = -0.989743

